# Fake News Detection - Advanced Notebook

This notebook contains a complete advanced fake news detection workflow:

- data loading and preprocessing
- feature engineering with TF-IDF
- comparison of multiple machine learning models
- evaluation on validation and test sets
- explainability with top words
- saving the best model for deployment


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

ARTIFACTS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)


In [2]:
def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def prepare_frame(df: pd.DataFrame, label: int) -> pd.DataFrame:
    frame = df.copy()
    frame["label"] = label
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["title"] = frame["title"].fillna("")
    frame["text"] = frame["text"].fillna("")
    frame["subject"] = frame["subject"].fillna("unknown")
    frame["content"] = (
        frame["title"].map(normalize_text)
        + " "
        + frame["subject"].map(normalize_text)
        + " "
        + frame["text"].map(normalize_text)
    )
    frame["word_count"] = frame["content"].str.split().str.len()
    frame["char_count"] = frame["content"].str.len()
    return frame


fake_df = pd.read_csv(PROJECT_DIR / "Fake.csv")
true_df = pd.read_csv(PROJECT_DIR / "True.csv")

dataset = pd.concat(
    [prepare_frame(fake_df, 1), prepare_frame(true_df, 0)],
    ignore_index=True,
)
dataset = dataset.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
dataset.head()


,title,text,subject,date,label,content,word_count,char_count
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,2017-02-13,1,ben stein calls out 9th circuit court committe...,189,1105
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,2017-04-05,0,trump drops steve bannon from national securit...,813,4780
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,2017-09-27,0,puerto rico expects u s to lift jones act ship...,324,1861
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,2017-05-22,1,oops trump just accidentally confirmed he leak...,201,1273
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,2016-06-24,0,donald trump heads for scotland to reopen a go...,555,3135


In [3]:
dataset_overview = {
    "rows": int(len(dataset)),
    "fake_rows": int(dataset["label"].sum()),
    "true_rows": int((dataset["label"] == 0).sum()),
    "avg_word_count": round(float(dataset["word_count"].mean()), 2),
    "median_word_count": round(float(dataset["word_count"].median()), 2),
    "subjects": dataset["subject"].value_counts().head(10).to_dict(),
}

profile = dataset.groupby("label")[["word_count", "char_count"]].agg(["mean", "median"])
profile.to_csv(REPORTS_DIR / "dataset_profile.csv")

pd.Series(dataset_overview)


rows                                                             44898
fake_rows                                                        23481
true_rows                                                        21417
avg_word_count                                                  429.72
median_word_count                                                385.0
subjects             {'politicsNews': 11272, 'worldnews': 10145, 'N...
dtype: object

In [4]:
train_df, temp_df = train_test_split(
    dataset,
    test_size=0.30,
    stratify=dataset["label"],
    random_state=RANDOM_STATE,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_STATE,
)

X_train, y_train = train_df["content"], train_df["label"]
X_val, y_val = val_df["content"], val_df["label"]
X_test, y_test = test_df["content"], test_df["label"]

print("Train shape:", X_train.shape[0])
print("Validation shape:", X_val.shape[0])
print("Test shape:", X_test.shape[0])


Train shape: 31428
Validation shape: 6735
Test shape: 6735


In [5]:
def make_vectorizer() -> TfidfVectorizer:
    return TfidfVectorizer(
        stop_words="english",
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
    )


models = {
    "logistic_regression": Pipeline(
        [
            ("tfidf", make_vectorizer()),
            ("model", LogisticRegression(max_iter=2000)),
        ]
    ),
    "linear_svc": Pipeline(
        [
            ("tfidf", make_vectorizer()),
            ("model", LinearSVC(C=1.0)),
        ]
    ),
    "multinomial_nb": Pipeline(
        [
            ("tfidf", make_vectorizer()),
            ("model", MultinomialNB(alpha=0.3)),
        ]
    ),
    "sgd_classifier": Pipeline(
        [
            ("tfidf", make_vectorizer()),
            ("model", SGDClassifier(loss="log_loss", random_state=RANDOM_STATE)),
        ]
    ),
}


def get_score_signal(model: Pipeline, texts: pd.Series) -> np.ndarray:
    estimator = model.named_steps["model"]
    if hasattr(estimator, "predict_proba"):
        return model.predict_proba(texts)[:, 1]
    return model.decision_function(texts)


def evaluate_model(model: Pipeline, texts: pd.Series, labels: pd.Series) -> dict[str, float]:
    predictions = model.predict(texts)
    scores = get_score_signal(model, texts)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "roc_auc": roc_auc_score(labels, scores),
    }


In [6]:
comparison_rows = []
trained_models = {}

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    metrics = evaluate_model(pipeline, X_val, y_val)
    comparison_rows.append({"model": model_name, **metrics})
    trained_models[model_name] = pipeline

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by=["f1", "roc_auc", "accuracy"], ascending=False
)
comparison_df.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
comparison_df


,model,accuracy,precision,recall,f1,roc_auc
1,linear_svc,0.998812,0.998581,0.999148,0.998865,0.999995
0,logistic_regression,0.995249,0.996303,0.994605,0.995453,0.999802
3,sgd_classifier,0.990794,0.994568,0.987791,0.991168,0.999503
2,multinomial_nb,0.966741,0.969801,0.966496,0.968146,0.993683


In [7]:
best_model_name = str(comparison_df.iloc[0]["model"])
best_model = trained_models[best_model_name]

test_predictions = best_model.predict(X_test)
test_scores = get_score_signal(best_model, X_test)

test_metrics = {
    "accuracy": accuracy_score(y_test, test_predictions),
    "precision": precision_score(y_test, test_predictions),
    "recall": recall_score(y_test, test_predictions),
    "f1": f1_score(y_test, test_predictions),
    "roc_auc": roc_auc_score(y_test, test_scores),
}

summary = {
    "best_model": best_model_name,
    "validation_leaderboard": comparison_df.to_dict(orient="records"),
    "test_metrics": test_metrics,
    "dataset_overview": dataset_overview,
}

with open(REPORTS_DIR / "summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

report = classification_report(y_test, test_predictions, target_names=["true", "fake"], digits=4)
with open(REPORTS_DIR / "classification_report.txt", "w", encoding="utf-8") as file:
    file.write(report)

confusion = confusion_matrix(y_test, test_predictions)
pd.DataFrame(
    confusion,
    index=["actual_true", "actual_fake"],
    columns=["pred_true", "pred_fake"],
).to_csv(REPORTS_DIR / "confusion_matrix.csv")

pd.Series(test_metrics)


accuracy     0.999109
precision    0.999432
recall       0.998865
f1           0.999148
roc_auc      0.999994
dtype: float64

In [8]:
def extract_top_terms(model: Pipeline, top_n: int = 20) -> tuple[pd.DataFrame, pd.DataFrame]:
    estimator = model.named_steps["model"]
    if not hasattr(estimator, "coef_"):
        empty = pd.DataFrame(columns=["term", "weight"])
        return empty, empty

    feature_names = model.named_steps["tfidf"].get_feature_names_out()
    coefficients = estimator.coef_[0]
    fake_indices = np.argsort(coefficients)[-top_n:][::-1]
    true_indices = np.argsort(coefficients)[:top_n]

    fake_terms = pd.DataFrame(
        {"term": feature_names[fake_indices], "weight": coefficients[fake_indices]}
    )
    true_terms = pd.DataFrame(
        {"term": feature_names[true_indices], "weight": coefficients[true_indices]}
    )
    return fake_terms, true_terms


top_fake_terms, top_true_terms = extract_top_terms(best_model)
top_fake_terms.to_csv(REPORTS_DIR / "top_fake_terms.csv", index=False)
top_true_terms.to_csv(REPORTS_DIR / "top_true_terms.csv", index=False)

print("Top fake-news terms")
display(top_fake_terms.head(10))
print("Top true-news terms")
display(top_true_terms.head(10))


Top fake-news terms


,term,weight
0,politics,5.584919
1,news,4.557596
2,left news,3.432051
3,government news,3.205084
4,read,2.560393
5,featured image,2.422911
6,featured,2.314237
7,image,2.300359
8,video,2.174864
9,gop,1.961944


Top true-news terms


,term,weight
0,reuters,-10.397564
1,politicsnews,-8.665878
2,worldnews,-6.555176
3,said,-3.878344
4,washington reuters,-3.092980
5,politicsnews reuters,-2.981920
6,politicsnews washington,-2.923844
7,president donald,-1.753027
8,reuters president,-1.659312
9,nov,-1.456749


In [9]:
artifact_bundle = {
    "model": best_model,
    "best_model_name": best_model_name,
    "label_mapping": {"0": "true", "1": "fake"},
}
joblib.dump(artifact_bundle, ARTIFACTS_DIR / "best_model.joblib")

print(f"Best model saved: {best_model_name}")
print("Artifacts saved to:", ARTIFACTS_DIR)
print("Reports saved to:", REPORTS_DIR)


Best model saved: linear_svc
Artifacts saved to: c:\Users\AG Computers\OneDrive\Documents\fake news detection project\artifacts
Reports saved to: c:\Users\AG Computers\OneDrive\Documents\fake news detection project\reports


In [10]:
sample_title = "Government announces major election reform"
sample_subject = "politics"
sample_text = "Officials said the policy was discussed in parliament and will be reviewed by committees before final approval."

sample_input = normalize_text(f"{sample_title} {sample_subject} {sample_text}")
prediction = int(best_model.predict([sample_input])[0])

estimator = best_model.named_steps["model"]
if hasattr(estimator, "predict_proba"):
    fake_probability = float(best_model.predict_proba([sample_input])[0][1])
else:
    raw_score = float(best_model.decision_function([sample_input])[0])
    fake_probability = 1.0 / (1.0 + np.exp(-raw_score))

print("Prediction:", "fake" if prediction == 1 else "true")
print("Fake probability:", round(fake_probability, 4))


Prediction: fake
Fake probability: 0.6068
